# CPT-LFR — E3_Control_LFR
**T4 GPU required.** Mount `cpt-datasets`.

**E3** — Curriculum only. Raw C4 + LFR ordering + replay buffer. Isolates curriculum effect vs E1.

**Token budget:** 10M tokens (~610 steps)  
**Estimated time:** ~16h @ 170 tok/s — likely needs 2 Kaggle sessions  
**Resume:** Re-run notebook after session timeout — picks up automatically from last checkpoint

**After completing:** save output as `cpt-e3`

In [1]:

!pip install --no-cache-dir torch==2.2.2 torchvision torchaudio \
--index-url https://download.pytorch.org/whl/cu121 -q

!pip install --no-cache-dir transformers==4.40.2 accelerate==0.30.1 \
bitsandbytes==0.45.5 peft==0.10.0 datasets -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 757.2/757.2 MB 258.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 260.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 318.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 239.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 296.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 233.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 261.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 189.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 267.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 312.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 MB 130.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 248.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# CELL 2 — GPU check + HF login
import torch
assert torch.cuda.is_available(), "Enable T4 GPU in notebook settings"
gpu  = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
cc   = torch.cuda.get_device_capability(0)
print(f"GPU  : {gpu}")
print(f"VRAM : {vram:.1f} GB")
print(f"CC   : {cc[0]}.{cc[1]}  (T4=7.5, using SDPA — FA2 needs CC≥8.0)")
from huggingface_hub import login
login("HF_TOKEN")
print("HF   : ✅ logged in")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py", line 37, in <module>
    ColabKernelApp.launch_instance()
  File "/usr/local/lib/python3.12/dist-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/usr/local/lib/python3.12/dist-packages/ipykernel/kernelapp.py", line 712, in start
    self.io_loop.start()
  File "/usr/local/lib/python3.12/dist-packag

GPU  : Tesla T4
VRAM : 15.6 GB
CC   : 7.5  (T4=7.5, using SDPA — FA2 needs CC≥8.0)
HF   : ✅ logged in


In [3]:
# CELL 3 — Configuration
# TOKEN_BUDGET = 10M tokens
# At ~170 tok/s (r=64 LoRA on T4): ~16h per experiment
# Kaggle sessions cap at 12h → checkpoint resume is essential
# Structure: one experiment per notebook, resume picks up mid-run if session times out

import os, torch, random, gc
import numpy as np

DEVICE    = "cuda"
SEED      = 42
MODEL_ID  = "mistralai/Mistral-7B-v0.1"

# LoRA — r=64 for CPT knowledge capacity (was r=16 → 0.576% params → too thin)
LORA_R       = 64
LORA_ALPHA   = 128
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj","k_proj","v_proj","o_proj",
                 "gate_proj","up_proj","down_proj","embed_tokens"]

# Token budget + derived training dimensions
SEQ_LEN          = 1024
BATCH_SIZE       = 2
GRAD_ACCUM       = 8
TOKEN_BUDGET     = 10_000_000           # 10M tokens
LR               = 2e-4
WARMUP_RATIO     = 0.03
MAX_GRAD_NORM    = 1.0

TOKENS_PER_STEP  = BATCH_SIZE * SEQ_LEN * GRAD_ACCUM   # 16,384
TOTAL_STEPS      = TOKEN_BUDGET // TOKENS_PER_STEP      # ~610 steps
EVAL_STEPS       = max(50, TOTAL_STEPS // 10)           # 10 eval points (~61 steps apart)
SAVE_STEPS       = EVAL_STEPS                           # checkpoint every eval

# Replay buffer
REPLAY_STORE_FRAC  = 0.20
REPLAY_INJECT_FRAC = 0.50

CKPT_DIR   = "/kaggle/working/experiments"
RESUME_DIR = "/kaggle/input/datasets/preetam026/cpt-e3/experiments"  # mounted cpt-e3 dataset
os.makedirs(CKPT_DIR, exist_ok=True)

def find_resume_state(exp_id):
    """Return path to state.pt from a previous session, or None if starting fresh.
    Checks the mounted input dataset first, then the current working dir."""
    for base in [RESUME_DIR, CKPT_DIR]:
        p = os.path.join(base, exp_id, "state.pt")
        if os.path.exists(p):
            print(f"  ↻ Resume state found: {p}")
            return p, base
    return None, None
os.makedirs(CKPT_DIR, exist_ok=True)

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def find_file(fname):
    for cand in [
        f"/kaggle/input/cpt-datasets/{fname}",
        f"/kaggle/input/cpt-datasets/working/{fname}",
        f"/kaggle/working/{fname}",
        f"/kaggle/input/datasets/preetam026/cpt-datasets/{fname}",
    ]:
        if os.path.exists(cand): return cand
    raise FileNotFoundError(f"{fname} not found — mount cpt-datasets")

# ── Time estimate ──────────────────────────────────────────────────
est_tps   = 170   # conservative estimate for r=64 on T4
est_hours = TOKEN_BUDGET / est_tps / 3600
print(f"Model          : {MODEL_ID}")
print(f"LoRA r         : {LORA_R}  targets: {len(LORA_TARGETS)} modules")
print(f"Token budget   : {TOKEN_BUDGET/1e6:.0f}M")
print(f"Tokens/step    : {TOKENS_PER_STEP:,}  ({BATCH_SIZE}×{SEQ_LEN}×{GRAD_ACCUM})")
print(f"Total steps    : ~{TOTAL_STEPS}")
print(f"Eval every     : {EVAL_STEPS} steps  (10 eval points total)")
print(f"Est. time @ {est_tps} tok/s: {est_hours:.1f}h")
print(f"⚠️  Likely exceeds one 12h Kaggle session — RESUME IS BUILT IN")
print(f"   If session times out: re-run notebook, it will resume from last checkpoint")

Model          : mistralai/Mistral-7B-v0.1
LoRA r         : 64  targets: 8 modules
Token budget   : 10M
Tokens/step    : 16,384  (2×1024×8)
Total steps    : ~610
Eval every     : 61 steps  (10 eval points total)
Est. time @ 170 tok/s: 16.3h
⚠️  Likely exceeds one 12h Kaggle session — RESUME IS BUILT IN
   If session times out: re-run notebook, it will resume from last checkpoint


In [4]:
# CELL 4 — Load datasets
import json

def load_jsonl(fname, label=""):
    data = [json.loads(l) for l in open(find_file(fname))]
    print(f"  ✅ {label or fname}: {len(data):,}")
    return data

print("Loading datasets...")
control_data  = load_jsonl("control_cpt.jsonl",  "control")
golden_data   = load_jsonl("golden_cpt.jsonl",   "golden")
val_neutral   = [d["text"] for d in load_jsonl("val_neutral.jsonl",   "val_neutral")]
val_indomain  = [d["text"] for d in load_jsonl("val_indomain.jsonl",  "val_indomain")]
val_ood       = [d["text"] for d in load_jsonl("val_ood.jsonl",       "val_ood")]
tier_val_docs = load_jsonl("tier_validation.jsonl", "tier_val")

for label, data in [("control", control_data), ("golden", golden_data)]:
    ph = [d["lfr_phase"] for d in data]
    e,m,h = ph.count("easy"),ph.count("medium"),ph.count("hard")
    assert e>0 and m>0 and h>0
    print(f"  {label} LFR: easy={e} medium={m} hard={h}")

Loading datasets...
  ✅ control: 10,000
  ✅ golden: 10,000
  ✅ val_neutral: 300
  ✅ val_indomain: 300
  ✅ val_ood: 300
  ✅ tier_val: 60
  control LFR: easy=3335 medium=3333 hard=3332
  golden LFR: easy=3338 medium=3329 hard=3333


In [5]:
# CELL 5 — Dataset, eval, model builder
import torch, math, random, gc
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForCausalLM,
                          BitsAndBytesConfig, get_cosine_schedule_with_warmup)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.optim import AdamW
from torch.amp import autocast
from torch.cuda.amp import GradScaler

TOK = AutoTokenizer.from_pretrained(MODEL_ID)
TOK.pad_token = TOK.eos_token
TOK.padding_side = "right"
print(f"Tokenizer: vocab={TOK.vocab_size:,}")

class CPTDataset(Dataset):
    def __init__(self, docs, tokeniser, seq_len):
        self.samples = []
        for doc in docs:
            text = doc["text"] if isinstance(doc,dict) else doc
            enc  = tokeniser(text, truncation=True, max_length=seq_len,
                             padding="max_length", return_tensors="pt")
            ids  = enc["input_ids"][0]
            mask = enc["attention_mask"][0]
            if mask.sum().item() < seq_len // 4: continue
            self.samples.append({
                "input_ids": ids, "attention_mask": mask, "labels": ids.clone()})
    def __len__(self):           return len(self.samples)
    def __getitem__(self, i):    return self.samples[i]

def eval_3way(model, val_neutral, val_indomain, val_ood, tokeniser, seq_len, device, n=60):
    """Evaluate on all three val splits. Returns (neutral, indomain, ood) as (loss, ppl) pairs."""
    def _ppl(texts):
        model.eval(); total_loss=0.0; total_tok=0
        sample = random.sample(texts, min(n, len(texts)))
        with torch.no_grad():
            for text in sample:
                enc = tokeniser(text, return_tensors="pt", truncation=True,
                                max_length=seq_len, padding=False).to(device)
                n_tok = enc["attention_mask"].sum().item()
                if n_tok < 10: continue
                with autocast("cuda", dtype=torch.float16):
                    out = model(**enc, labels=enc["input_ids"])
                total_loss += out.loss.item() * n_tok
                total_tok  += n_tok
        model.train()
        avg = total_loss / max(total_tok, 1)
        return round(avg, 4), round(math.exp(min(avg, 20)), 4)
    n_r, p_r = _ppl(val_neutral)
    n_i, p_i = _ppl(val_indomain)
    n_o, p_o = _ppl(val_ood)
    return (n_r,p_r), (n_i,p_i), (n_o,p_o)

def build_model():
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb, device_map="auto", attn_implementation="sdpa")
    base = prepare_model_for_kbit_training(base, use_gradient_checkpointing=True)
    model = get_peft_model(base, LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGETS, bias="none", task_type="CAUSAL_LM"))
    tr, tot = model.get_nb_trainable_parameters()
    print(f"  Trainable: {tr:,} ({100*tr/tot:.3f}%)  "
          f"VRAM: {torch.cuda.max_memory_allocated(0)/1e9:.2f}GB")
    return model

print("Core helpers ready ✅")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Tokenizer: vocab=32,000
Core helpers ready ✅


In [6]:
# CELL 6 — Training function WITH CHECKPOINT RESUME
# If a session times out mid-experiment, re-run the notebook.
# It will detect the latest checkpoint and resume from there automatically.

import csv, json, time, os, random, torch
from torch.utils.data import DataLoader
from torch.amp import autocast
from torch.cuda.amp import GradScaler

def run_experiment(exp_id, data, use_lfr,
                   val_neutral, val_indomain, val_ood, desc=""):
    """
    Single-experiment training function with full checkpoint resume.
    
    Resume logic:
    1. Check for state.pt in experiment directory
    2. If found: load model weights + optimizer state + step counters
    3. Continue from exact step — tokens_done, global_step, replay_buffer all restored
    4. Skips phases already completed
    
    This means a session timeout is not a failure — just restart.
    """
    print(f"\n{'='*65}")
    print(f"  {exp_id}")
    print(f"  {desc}")
    print(f"  use_lfr={use_lfr}  budget={TOKEN_BUDGET/1e6:.0f}M tokens")
    print(f"{'='*65}")

    ckpt_dir   = os.path.join(CKPT_DIR, exp_id)
    os.makedirs(ckpt_dir, exist_ok=True)
    train_csv  = os.path.join(ckpt_dir, "train_log.csv")
    eval_csv   = os.path.join(ckpt_dir, "eval_log.csv")
    state_path = os.path.join(ckpt_dir, "state.pt")
    result_path= os.path.join(ckpt_dir, "result.json")

    # ── Already finished? ─────────────────────────────────────────
    if os.path.exists(result_path):
        result = json.load(open(result_path))
        if result.get("tokens_processed", 0) >= TOKEN_BUDGET * 0.95:
            print(f"  ✅ Already complete ({result['tokens_processed']/1e6:.1f}M tokens)")
            print(f"     Neutral PPL={result['final_ppl_neutral']}  "
                  f"In-domain={result['final_ppl_indomain']}  OOD={result['final_ppl_ood']}")
            return result

    # ── Build phase plan ──────────────────────────────────────────
    replay_buffer = []
    if use_lfr:
        easy_d   = [d for d in data if d.get("lfr_phase") == "easy"]
        medium_d = [d for d in data if d.get("lfr_phase") == "medium"]
        hard_d   = [d for d in data if d.get("lfr_phase") == "hard"]
        assert len(easy_d)>0 and len(medium_d)>0 and len(hard_d)>0
        phase_plan = [("LEARN",easy_d), ("FOCUS",medium_d), ("REVIEW",hard_d)]
        print(f"  LFR: easy={len(easy_d)} medium={len(medium_d)} hard={len(hard_d)}")
    else:
        shuffled = list(data); random.shuffle(shuffled)
        phase_plan = [("TRAIN", shuffled)]

    # ── Load or initialise model ───────────────────────────────────
    model    = build_model()
    opt, sch = build_opt_sch(model)
    scaler   = GradScaler()

    # Resume state
    global_step  = 0
    tokens_done  = 0
    phase_idx    = 0
    ppl_history  = []

    if os.path.exists(state_path):
        print(f"\n  ↻ RESUMING from checkpoint: {state_path}")
        state = torch.load(state_path, map_location="cpu")
        global_step  = state["global_step"]
        tokens_done  = state["tokens_done"]
        phase_idx    = state.get("phase_idx", 0)
        ppl_history  = state.get("ppl_history", [])
        replay_buffer= state.get("replay_buffer", [])
        opt.load_state_dict(state["opt"])
        sch.load_state_dict(state["sched"])
        # Reload adapter weights from last checkpoint
        last_ckpt = os.path.join(ckpt_dir, f"step{global_step:04d}")
        if os.path.isdir(last_ckpt):
            from peft import set_peft_model_state_dict
            import safetensors.torch as st
            adapter_path = os.path.join(last_ckpt, "adapter_model.safetensors")
            if os.path.exists(adapter_path):
                adapter_weights = st.load_file(adapter_path)
                set_peft_model_state_dict(model, adapter_weights)
                print(f"     Adapter weights loaded from step {global_step}")
        print(f"     Resumed at step {global_step} | {tokens_done/1e6:.2f}M tokens done")
        print(f"     Replay buffer: {len(replay_buffer)} docs")
    else:
        # New run — initialise CSV headers
        with open(train_csv,"w",newline="") as f:
            csv.writer(f).writerow(["step","tokens_M","loss","lr",
                                    "tps","vram_gb","phase"])
        with open(eval_csv,"w",newline="") as f:
            csv.writer(f).writerow(["step","tokens_M",
                                    "ppl_neutral","loss_neutral",
                                    "ppl_indomain","loss_indomain",
                                    "ppl_ood","loss_ood"])

    # ── Main training loop ─────────────────────────────────────────
    t0 = time.time()
    model.train()

    for p_idx, (phase_name, phase_data) in enumerate(phase_plan):
        if p_idx < phase_idx:
            print(f"  ── Phase {phase_name}: skipping (already completed in previous session)")
            continue
        if tokens_done >= TOKEN_BUDGET:
            break
        if len(phase_data) == 0:
            print(f"  ── Phase {phase_name}: SKIPPED (0 docs)")
            continue

        # Inject replay buffer into REVIEW
        if phase_name == "REVIEW" and len(replay_buffer) > 0:
            n_inj  = int(len(phase_data) * REPLAY_INJECT_FRAC)
            n_inj  = min(n_inj, len(replay_buffer))
            inject = random.sample(replay_buffer, n_inj)
            phase_data = phase_data + inject
            random.shuffle(phase_data)
            print(f"\n  ── Phase REVIEW: {len(phase_data):,} docs "
                  f"({n_inj} replay injected)")
        else:
            print(f"\n  ── Phase {phase_name}: {len(phase_data):,} docs")

        ds     = CPTDataset(phase_data, TOK, SEQ_LEN)
        if len(ds) == 0:
            print(f"     All docs too short — skipping")
            continue
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                            drop_last=True, num_workers=0)

        learn_losses = [] if phase_name == "LEARN" else None
        micro_step   = 0
        opt.zero_grad()

        for batch in loader:
            if tokens_done >= TOKEN_BUDGET:
                break
            t1   = time.time()
            ids  = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            lbls = batch["labels"].to(DEVICE)
            btok = mask.sum().item()

            with autocast("cuda", dtype=torch.float16):
                out  = model(input_ids=ids, attention_mask=mask, labels=lbls)
                loss = out.loss / GRAD_ACCUM
            scaler.scale(loss).backward()

            if learn_losses is not None:
                doc_idx = micro_step % len(phase_data)
                learn_losses.append((loss.item()*GRAD_ACCUM, phase_data[doc_idx]))

            micro_step  += 1
            tokens_done += btok

            # ── Optimizer step boundary — ONLY place eval/save fires ──
            if micro_step % GRAD_ACCUM == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model.parameters()),
                    MAX_GRAD_NORM)
                scaler.step(opt); scaler.update()
                opt.zero_grad(); sch.step()
                global_step += 1
                raw_loss = loss.item() * GRAD_ACCUM
                tps      = btok / max(time.time()-t1, 1e-6)
                vram     = torch.cuda.max_memory_allocated(0) / 1e9
                lr_now   = sch.get_last_lr()[0]
                with open(train_csv,"a",newline="") as f:
                    csv.writer(f).writerow([
                        global_step, round(tokens_done/1e6,3),
                        round(raw_loss,4), round(lr_now,8),
                        round(tps,1), round(vram,3), phase_name])

                # ── Eval + save checkpoint ────────────────────────
                if global_step % EVAL_STEPS == 0:
                    (l_n,p_n),(l_i,p_i),(l_o,p_o) = eval_3way(
                        model, val_neutral, val_indomain, val_ood,
                        TOK, SEQ_LEN, DEVICE)
                    ppl_history.append((global_step, p_n, p_i, p_o))
                    with open(eval_csv,"a",newline="") as f:
                        csv.writer(f).writerow([
                            global_step, round(tokens_done/1e6,3),
                            p_n,l_n, p_i,l_i, p_o,l_o])
                    elapsed  = time.time()-t0
                    done_pct = tokens_done/TOKEN_BUDGET*100
                    eta_h    = (TOKEN_BUDGET-tokens_done)/max(tps,1)/3600
                    print(f"    step {global_step:4d} | {tokens_done/1e6:5.2f}M tok ({done_pct:.0f}%) | "
                          f"loss {raw_loss:.4f} | "
                          f"PPL n={p_n:.3f} i={p_i:.3f} o={p_o:.3f} | "
                          f"{tps:.0f} tok/s | ETA {eta_h:.1f}h")
                    # Save adapter checkpoint
                    ckpt_path = os.path.join(ckpt_dir, f"step{global_step:04d}")
                    model.save_pretrained(ckpt_path)
                    TOK.save_pretrained(ckpt_path)
                    # Save full resume state
                    torch.save({
                        "global_step"  : global_step,
                        "tokens_done"  : tokens_done,
                        "phase_idx"    : p_idx,
                        "ppl_history"  : ppl_history,
                        "replay_buffer": replay_buffer,
                        "opt"          : opt.state_dict(),
                        "sched"        : sch.state_dict(),
                    }, state_path)
                    print(f"     ✅ Checkpoint + resume state saved")
                    # Early stop: neutral PPL worsening 3 consecutive evals
                    if len(ppl_history) >= 3:
                        recent = [x[1] for x in ppl_history[-3:]]
                        if all(recent[i] >= recent[i-1] for i in range(1,3)):
                            print("  Early stop — neutral PPL not improving")
                            tokens_done = TOKEN_BUDGET  # breaks outer loop too
                            break
                elif global_step % 25 == 0:
                    elapsed_h = (time.time()-t0)/3600
                    eta_h = (TOKEN_BUDGET-tokens_done)/max(tps,1)/3600
                    print(f"    step {global_step:4d} | {tokens_done/1e6:5.2f}M | "
                          f"loss {raw_loss:.4f} | {tps:.0f} tok/s | "
                          f"elapsed {elapsed_h:.1f}h  ETA {eta_h:.1f}h")

        # Build replay buffer at end of LEARN
        if phase_name == "LEARN" and learn_losses:
            learn_losses.sort(key=lambda x: x[0], reverse=True)
            n_store = int(len(learn_losses) * REPLAY_STORE_FRAC)
            replay_buffer = [doc for _, doc in learn_losses[:n_store]]
            print(f"  ↳ Replay buffer: {len(replay_buffer)} high-loss docs stored")

    # ── Final eval + result ────────────────────────────────────────
    (l_n,p_n),(l_i,p_i),(l_o,p_o) = eval_3way(
        model, val_neutral, val_indomain, val_ood, TOK, SEQ_LEN, DEVICE)
    total_time = time.time()-t0
    tr, tot    = model.get_nb_trainable_parameters()
    result = {
        "exp_id"             : exp_id,
        "desc"               : desc,
        "use_lfr"            : use_lfr,
        "model_id"           : MODEL_ID,
        "lora_r"             : LORA_R,
        "token_budget"       : TOKEN_BUDGET,
        "tokens_processed"   : tokens_done,
        "total_steps"        : global_step,
        "final_ppl_neutral"  : p_n,   "final_loss_neutral"  : l_n,
        "final_ppl_indomain" : p_i,   "final_loss_indomain" : l_i,
        "final_ppl_ood"      : p_o,   "final_loss_ood"      : l_o,
        "ppl_history"        : ppl_history,
        "replay_buffer_size" : len(replay_buffer),
        "avg_tps"            : round(tokens_done/max(total_time,1),1),
        "peak_vram_gb"       : round(torch.cuda.max_memory_allocated(0)/1e9,3),
        "total_time_min"     : round(total_time/60,2),
        "trainable_pct"      : round(100*tr/tot,4),
    }
    with open(result_path,"w") as f: json.dump(result, f, indent=2)
    model.save_pretrained(os.path.join(ckpt_dir,"final_adapter"))
    TOK.save_pretrained(os.path.join(ckpt_dir,"final_adapter"))
    del model, opt, scaler; gc.collect(); torch.cuda.empty_cache()
    print(f"\n  ✅ {exp_id} COMPLETE")
    print(f"     Neutral PPL   : {p_n}")
    print(f"     In-domain PPL : {p_i}")
    print(f"     OOD PPL       : {p_o}")
    print(f"     Tokens        : {tokens_done/1e6:.2f}M")
    print(f"     Time          : {total_time/60:.1f}min")
    print(f"     Avg tok/s     : {result['avg_tps']}")
    return result

def build_opt_sch(model):
    opt = AdamW(filter(lambda p:p.requires_grad,model.parameters()),
                lr=LR,betas=(0.9,0.999),eps=1e-8,weight_decay=0.01)
    sched = get_cosine_schedule_with_warmup(opt,
        num_warmup_steps=int(TOTAL_STEPS*WARMUP_RATIO),
        num_training_steps=TOTAL_STEPS)
    return opt, sched

print("Training function (with resume) ready ✅")

Training function (with resume) ready ✅


In [7]:
# CELL 6b — Tier validation (run only for LFR experiments)
# Verifies easy < medium < hard by base model PPL before spending GPU budget

def run_tier_validation():
    import gc
    from transformers import AutoModelForCausalLM
    print("Tier validation — loading base model briefly (~60s)...")
    bm = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map="auto")
    bm.eval()

    def _quick_ppl(docs, n=20):
        total_loss=0.0; total_tok=0
        for d in docs[:n]:
            text = d["text"] if isinstance(d,dict) else d
            enc  = TOK(text,return_tensors="pt",truncation=True,
                       max_length=512,padding=False).to("cuda")
            n_tok = enc["attention_mask"].sum().item()
            if n_tok<10: continue
            with torch.no_grad(), autocast("cuda",dtype=torch.float16):
                out = bm(**enc,labels=enc["input_ids"])
            total_loss+=out.loss.item()*n_tok; total_tok+=n_tok
        avg=total_loss/max(total_tok,1)
        return round(math.exp(min(avg,20)),4)

    easy_docs  = [d for d in tier_val_docs if d["lfr_phase"]=="easy"]
    med_docs   = [d for d in tier_val_docs if d["lfr_phase"]=="medium"]
    hard_docs  = [d for d in tier_val_docs if d["lfr_phase"]=="hard"]
    p_e = _quick_ppl(easy_docs)
    p_m = _quick_ppl(med_docs)
    p_h = _quick_ppl(hard_docs)
    del bm; gc.collect(); torch.cuda.empty_cache()

    print(f"  LEARN  (easy)  PPL: {p_e}")
    print(f"  FOCUS  (medium) PPL: {p_m}")
    print(f"  REVIEW (hard)  PPL: {p_h}")
    ok = p_e < p_m < p_h
    print(f"  Ordering valid: {'✅ YES — curriculum is meaningful' if ok else '⚠️  NO — check difficulty scoring'}")
    return ok

print("Tier validation function ready ✅")

Tier validation function ready ✅


In [8]:
# ── RESTORE CHECKPOINT FROM cpt-e3 DATASET ────────────────────
import os, shutil, torch

# Your mounted dataset — check Settings → Data to confirm exact path
PREV_DATASET = "/kaggle/input/datasets/preetam026/cpt-e3-correct"

def restore_checkpoint(exp_id):
    exp_working   = os.path.join(CKPT_DIR, exp_id)
    state_working = os.path.join(exp_working, "state.pt")

    # Already restored this session — nothing to do
    if os.path.exists(state_working):
        state = torch.load(state_working, map_location="cpu")
        print(f"✅ Checkpoint in working dir — step {state['global_step']}, "
              f"{state['tokens_done']/1e6:.2f}M tokens done")
        return True

    # Search mounted dataset for state.pt matching this experiment
    found = None
    for root, dirs, files in os.walk(PREV_DATASET):
        if "state.pt" in files and exp_id in root:
            found = os.path.join(root, "state.pt")
            break

    if not found:
        print(f"No checkpoint found in {PREV_DATASET} — starting from scratch")
        return False

    # Copy entire experiment directory to working
    src_dir = os.path.dirname(found)
    os.makedirs(exp_working, exist_ok=True)
    for item in os.listdir(src_dir):
        s = os.path.join(src_dir, item)
        d = os.path.join(exp_working, item)
        if os.path.isfile(s):
            shutil.copy2(s, d)
        elif os.path.isdir(s) and not os.path.exists(d):
            shutil.copytree(s, d)

    state = torch.load(state_working, map_location="cpu")
    print(f"✅ Restored from {src_dir}")
    print(f"   Resuming at step {state['global_step']}, "
          f"{state['tokens_done']/1e6:.2f}M tokens done")
    return True

restore_checkpoint("E3_Control_LFR")

✅ Restored from /kaggle/input/datasets/preetam026/cpt-e3-correct/experiments/E3_Control_LFR
   Resuming at step 549, 4.17M tokens done


True

In [9]:
# Run tier validation before spending GPU budget
tier_ok = run_tier_validation()
if not tier_ok:
    print("⚠️  Tier ordering invalid — curriculum may not help. Continuing anyway.")
import sys, warnings
sys.setrecursionlimit(10000)
warnings.filterwarnings("ignore")

result = run_experiment(
    exp_id   = "E3_Control_LFR",
    data     = control_data,
    use_lfr  = True,
    val_neutral  = val_neutral,
    val_indomain = val_indomain,
    val_ood      = val_ood,
    desc     = "CPT curriculum: raw C4, LFR ordering + replay buffer, r=64 LoRA, 3-way eval, 10M tokens",
)


Tier validation — loading base model briefly (~60s)...


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

...

  LEARN  (easy)  PPL: 9.7747
  FOCUS  (medium) PPL: 7.7356
  REVIEW (hard)  PPL: 8.2574
  Ordering valid: ⚠️  NO — check difficulty scoring
⚠️  Tier ordering invalid — curriculum may not help. Continuing anyway.

  E3_Control_LFR
  CPT curriculum: raw C4, LFR ordering + replay buffer, r=64 LoRA, 3-way eval, 10M tokens
  use_lfr=True  budget=10M tokens
  LFR: easy=3335 medium=3333 hard=3332


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

  Trainable: 170,082,304 (2.295%)  VRAM: 2.50GB

  ↻ RESUMING from checkpoint: /kaggle/working/experiments/E3_Control_LFR/state.pt
     Adapter weights loaded from step 549
     Resumed at step 549 | 4.17M tokens done
     Replay buffer: 257 docs
  ── Phase LEARN: skipping (already completed in previous session)
  ── Phase FOCUS: skipping (already completed in previous session)

  ── Phase REVIEW: 3,589 docs (257 replay injected)


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


    step  550 |  4.17M | loss 0.8775 | 115 tok/s | elapsed 0.0h  ETA 14.0h
    step  575 |  4.35M | loss 0.5304 | 87 tok/s | elapsed 0.5h  ETA 18.1h
    step  600 |  4.53M | loss 0.6896 | 91 tok/s | elapsed 1.0h  ETA 16.8h
    step  610 |  4.60M tok (46%) | loss 1.1599 | PPL n=20.412 i=13.379 o=12.974 | 148 tok/s | ETA 10.1h
     ✅ Checkpoint + resume state saved
  Early stop — neutral PPL not improving

  ✅ E3_Control_LFR COMPLETE
     Neutral PPL   : 21.7241
     In-domain PPL : 14.6109
     OOD PPL       : 14.7974
     Tokens        : 10.00M
     Time          : 76.4min
     Avg tok/s     : 2180.4


In [10]:
import json, os, pprint

result_path = os.path.join(CKPT_DIR, "E3_Control_LFR", "result.json")
if os.path.exists(result_path):
    r = json.load(open(result_path))
    print("\n" + "="*60)
    print("E3_Control_LFR — FINAL RESULTS")
    print("="*60)
    print(f"  Neutral PPL   : {r['final_ppl_neutral']}")
    print(f"  In-domain PPL : {r['final_ppl_indomain']}")
    print(f"  OOD PPL       : {r['final_ppl_ood']}")
    print(f"  Tokens        : {r['tokens_processed']/1e6:.2f}M / {TOKEN_BUDGET/1e6:.0f}M")
    print(f"  Avg tok/s     : {r['avg_tps']}")
    print(f"  Peak VRAM     : {r['peak_vram_gb']} GB")
    print(f"  Total time    : {r['total_time_min']:.1f} min")
    print(f"  PPL history   : {r['ppl_history']}")
    if r.get('tokens_processed', 0) < TOKEN_BUDGET * 0.95:
        remaining = (TOKEN_BUDGET - r['tokens_processed']) / max(r['avg_tps'],1) / 3600
        print(f"\n  ⚠️  INCOMPLETE — {r['tokens_processed']/1e6:.1f}M / {TOKEN_BUDGET/1e6:.0f}M tokens")
        print(f"     Est. remaining: {remaining:.1f}h")
        print(f"     Re-run this notebook to resume automatically from step {r['total_steps']}")
    else:
        print("\n  ✅ COMPLETE")
        print(f"  Save output as Kaggle Dataset: 'cpt-e3'")



E3_Control_LFR — FINAL RESULTS
  Neutral PPL   : 21.7241
  In-domain PPL : 14.6109
  OOD PPL       : 14.7974
  Tokens        : 10.00M / 10M
  Avg tok/s     : 2180.4
  Peak VRAM     : 5.349 GB
  Total time    : 76.4 min
  PPL history   : [[61, 30.4934, 22.6536, 18.7394], [122, 35.6911, 17.4314, 19.3704], [183, 28.8753, 21.5631, 19.5629], [244, 28.6568, 17.8006, 16.8948], [305, 23.9512, 15.8641, 15.3466], [366, 22.2525, 16.1069, 14.2435], [427, 20.6208, 13.9263, 14.3343], [488, 18.778, 12.3089, 11.9222], [549, 20.0308, 13.4648, 13.4946], [610, 20.4121, 13.3787, 12.9742]]

  ✅ COMPLETE
  Save output as Kaggle Dataset: 'cpt-e3'
